# Chapter 3 · Representations and conversions

**Learning outcomes:** Inspect outside spaces and reference labels; distinguish an absent value from an unreadable value; interpret numbers and dates; and save converted values for later rules.

**Checks in this chapter:** CHK04 Outside spaces; CHK05 State codes; CHK06 Informative cause; CHK07–CHK08 Number readiness; CHK09–CHK10 Date readiness.

**How to run:** Open the extracted study-guide folder in VS Code. Run this notebook from the top.
Each notebook loads its own `Data/wildfires.csv`. Chapters 4 and 5 also load the saved results named
beside their loading code. Restarting a kernel does not delete these saved files.



## Block 1 · Load the source and recall the result pattern

**Objective:** Start this chapter with its own source table and the same reporting functions.

**Expectation:** A present value can be readable yet fail a later meaning rule; an absent value can prevent a particular comparison.

**Where it comes from:** The distinction between availability, representation and meaning.

**Objects used here:** `raw` will be the unchanged source DataFrame. The imported functions summarise counts and save evidence; they do not implement conditions.

**Method reference:** [read_csv()](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) · [value_counts()](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html)


In [ ]:
import pandas as pd
from pathlib import Path
from inspection_helpers import summarise_check, save_check

# Start from the same unchanged teaching file in every notebook.
# Paths are relative to the study-guide folder opened in VS Code.
DATASET_PATH = Path("Data/wildfires.csv")
raw = pd.read_csv(DATASET_PATH, low_memory=False)

# An empty table cannot show that values passed a check.
if len(raw) == 0:
    raise ValueError("No records were loaded. Check Data/wildfires.csv before continuing.")

print("Source file:", DATASET_PATH)
print("Imported records:", len(raw))
print("Columns:", raw.columns.tolist())


**Reporting reminder:** `summarise_check(ID, title, total, flagged, untested=0)` returns a
check dictionary. `save_check(check, evidence, finding, effect, next_action)` saves the three
result files. The functions' full code is in Chapter 2's study reference.

**Local variables:** Each block creates its own condition, count, evidence and result close
together. The converted-number and converted-date tables are saved at the end of their own
blocks. Chapter 4 reloads those files instead of depending on this kernel.

**Combining decisions:** `&` means both conditions must be True; `|` means at least one
must be True; `~` reverses True and False. Use parentheses around comparisons when
combining them. A comparison involving missing text can produce a missing decision.
`fillna(False)` makes the flag False in that case, while a separate Untested condition
still records the unavailable prerequisite. False for a flag does not automatically mean Passed.


## Block 2 · CHK04 · Make outside spaces visible

**Objective:** Find present names whose spelling changes when outside spaces are trimmed.

**Expectation:** A present name has no outside spaces under our classroom comparison convention. Oak meets it; " Oak " is Flagged.

**Where it comes from:** An agreed comparison convention. A spelling difference does not establish that records describe the same fire.

**Objects used here:** `space_demo` is a small example. In the source check below, names and trimmed_names will be parallel Series with the original row labels.

**Method reference:** [Series()](https://pandas.pydata.org/docs/reference/api/pandas.Series.html) · [astype()](https://pandas.pydata.org/docs/reference/api/pandas.Series.astype.html) · [str.strip()](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.strip.html) · [str.casefold()](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.casefold.html) · [fillna()](https://pandas.pydata.org/docs/reference/api/pandas.Series.fillna.html) · [loc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html) · [repr()](https://docs.python.org/3/library/functions.html#repr)


In [ ]:
# Quotes make otherwise hidden spaces visible in these invented examples.
space_demo = pd.Series(["Oak", " Oak ", "OAK", None], dtype="string")
demo_trimmed_names = space_demo.str.strip()
demo_name_comparison = pd.DataFrame({
    "Original with quotes": space_demo.map(repr),
    "Trimmed with quotes": demo_trimmed_names.map(repr)
})
display(demo_name_comparison)

# casefold also changes OAK to oak for comparison. It does not identify an incident.
print("Case-insensitive comparison:")
display(space_demo.str.strip().str.casefold())


**Read the result:** Trimming removes outside spaces; it leaves OAK uppercase. Casefold changes letter case for a separate comparison. A name containing only spaces would become empty after trimming.



**Pause and explain:** Why should we keep the original spelling beside the comparison value?

<details>
<summary>Suggested answer — open after discussing</summary>

The original explains what was recorded and why the check flagged it. The comparison is temporary evidence.

</details>


In [ ]:
# Use Pandas text values so missing names stay missing during text comparisons.
names = raw["fire_name"].astype("string")
trimmed_names = names.str.strip()
present_names = names.notna()
name_spelling_changed = names != trimmed_names

# Missing names cannot supply characters to compare. They are Untested here,
# although they were Flagged by the separate availability check CHK01.
names_with_spaces = (present_names & name_spelling_changed).fillna(False)
names_untested = ~present_names
check_spaces = summarise_check(
    "CHK04", "Name has no outside spaces", len(raw),
    names_with_spaces.sum(), names_untested.sum()
)

# Keep both Flagged and Untested records as complete evidence of this check's concerns.
space_concerns = names_with_spaces | names_untested
evidence_spaces = raw.loc[space_concerns, ["fire_name", "state"]].copy()
evidence_spaces["Comparison name"] = trimmed_names.loc[evidence_spaces.index]
evidence_spaces["Record outcome"] = "Flagged"
evidence_spaces.loc[names_untested.loc[evidence_spaces.index], "Record outcome"] = "Untested"
# repr adds visible quotes for the preview; the original name column remains unchanged.
evidence_spaces["Name with quotes"] = names.loc[evidence_spaces.index].map(repr)
display(pd.DataFrame([check_spaces]))
# Show a few of each outcome here: a simple head could show only missing names
# and hide the actual outside-space examples we are trying to explain.
display(evidence_spaces.loc[evidence_spaces["Record outcome"] == "Flagged"].head(5))
display(evidence_spaces.loc[evidence_spaces["Record outcome"] == "Untested"].head(3))
save_check(
    check_spaces, evidence_spaces,
    finding="Outside spaces or unavailable names need attention for the name comparison.",
    effect="Q4 may display or compare equivalent spellings differently; unnamed records still lack a label.",
    next_action="Review the original and comparison values. Agree a formatting treatment and handle absent names separately."
)


**Read the result:** The Flagged count concerns changed spellings; Untested concerns absent names. Comparison name aligns by original row label. Neither the source names nor raw have been trimmed.



**Pause and explain:** Why do we select trimmed_names using evidence_spaces.index?

<details>
<summary>Suggested answer — open after discussing</summary>

We need the comparison value for each selected original record, not the first values of the whole Series.

</details>


## Block 3 · CHK05–CHK06 · Compare categories with justified reference lists

**Objective:** Check state-code spellings and distinguish an informative reported cause from an undefined cause.

**Expectation:** For CHK05, a present state is in the supplied list. For CHK06, a present cause is in the informative-cause list.

**Where it comes from:** The state list is an explicit classroom scope for the extract. Cause labels follow the source edition; Missing/Undefined is permitted source text but cannot identify a reported cause for Q2.

**Objects used here:** `raw` is the source. state_reference and cause_reference will be loaded reference tables. Each condition and evidence table is created in the same cell where it is used.

**Method reference:** [read_csv()](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) · [value_counts()](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html) · [astype()](https://pandas.pydata.org/docs/reference/api/pandas.Series.astype.html) · [isin()](https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html) · [notna()](https://pandas.pydata.org/docs/reference/api/pandas.Series.notna.html) · [fillna()](https://pandas.pydata.org/docs/reference/api/pandas.Series.fillna.html) · [loc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html)


In [ ]:
# Seeing observed labels is useful, but observed labels are not the acceptance rule.
# Otherwise every value we found would automatically be accepted.
display(raw["state"].value_counts(dropna=False))
display(raw["fire_cause"].value_counts(dropna=False))


**Examples:** `CA` meets the supplied state-code rule; `ZZ` and `ca` do not. The source may
still contain a perfectly understandable `ca`; we would investigate its formatting rather
than immediately delete it. This rule checks exact source spellings before any correction.

`Lightning` meets the informative-cause expectation. `Missing/Undefined` does not identify
a reported cause even though it is a legitimate source label. The issue concerns Q2's information,
not a claim that the database has used an illegal label.


In [ ]:
state_reference = pd.read_csv("Data/State_Code_Reference.csv")
allowed_state_codes = state_reference["code"]
state_values = raw["state"].astype("string")
state_available = state_values.notna()
state_matches_reference = state_values.isin(allowed_state_codes)
state_flagged = (state_available & ~state_matches_reference).fillna(False)
state_untested = ~state_available
check_states = summarise_check(
    "CHK05", "State code matches the reference list", len(raw),
    state_flagged.sum(), state_untested.sum()
)
evidence_states = raw.loc[state_flagged | state_untested, ["fire_name", "state", "fire_year"]].copy()
evidence_states["Record outcome"] = "Flagged"
evidence_states.loc[state_untested.loc[evidence_states.index], "Record outcome"] = "Untested"
display(pd.DataFrame([check_states]))
display(evidence_states.head(5))
save_check(
    check_states, evidence_states,
    finding="Some state values are absent or do not match the supplied code list.",
    effect="Q1 may form incorrect state groups or lack a state group for some records.",
    next_action="Review source codes, spelling and the agreed geographical scope before correcting or excluding values."
)


In [ ]:
cause_reference = pd.read_csv("Data/Informative_Cause_Reference.csv")
informative_cause_labels = cause_reference["cause"]
cause_values = raw["fire_cause"].astype("string")
cause_available = cause_values.notna()
cause_is_informative = cause_values.isin(informative_cause_labels)
cause_flagged = (cause_available & ~cause_is_informative).fillna(False)
cause_untested = ~cause_available
check_causes = summarise_check(
    "CHK06", "Informative reported cause", len(raw),
    cause_flagged.sum(), cause_untested.sum()
)
evidence_causes = raw.loc[cause_flagged | cause_untested, ["fire_name", "fire_cause", "fire_size"]].copy()
evidence_causes["Record outcome"] = "Flagged"
evidence_causes.loc[cause_untested.loc[evidence_causes.index], "Record outcome"] = "Untested"
display(pd.DataFrame([check_causes]))
display(evidence_causes.head(5))
save_check(
    check_causes, evidence_causes,
    finding="Some cause labels do not supply an informative reported cause for the planned comparison.",
    effect="Q2 must explain undefined or unrecognised causes when comparing size across reported causes.",
    next_action="Review the labels. Keep legitimate undefined causes visible as a separate limitation; do not guess a cause."
)


**Read the result:** Both summaries distinguish present-but-unsuitable labels from unavailable labels. Reference lists were supplied independently of the observed frequencies.



**Pause and explain:** Can we change Missing/Undefined to Lightning because Lightning is a common cause?

<details>
<summary>Suggested answer — open after discussing</summary>

No. Frequency does not establish the cause of an individual record.

</details>


## Block 4 · CHK07–CHK08 · Interpret numerical values

**Objective:** Find present values that cannot supply an ordinary number and save readable numbers for meaning checks.

**Expectation:** A present value converts to an ordinary number. Readable zero, negative and fractional values can pass this representation check.

**Where it comes from:** Numerical comparisons need numbers. Whether a number is acceptable for size or year is a separate expectation in Chapter 4.

**Objects used here:** `number_demo` is a small example. In the source cells, number_values is a separate two-column DataFrame holding interpreted size and year values, aligned with raw.

**Method reference:** [Series()](https://pandas.pydata.org/docs/reference/api/pandas.Series.html) · [to_numeric()](https://pandas.pydata.org/docs/reference/api/pandas.to_numeric.html) · [replace()](https://pandas.pydata.org/docs/reference/api/pandas.Series.replace.html) · [float()](https://docs.python.org/3/library/functions.html#float) · [isna()](https://pandas.pydata.org/docs/reference/api/pandas.Series.isna.html) · [notna()](https://pandas.pydata.org/docs/reference/api/pandas.Series.notna.html) · [loc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html) · [to_csv()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html)


In [ ]:
number_demo = pd.Series(["12.5", "-2", "unknown", None, "inf"], dtype="string")
demo_numbers = pd.to_numeric(number_demo, errors="coerce")
# Infinity is not an ordinary physical area or calendar year. Make it unavailable
# for later comparisons, just as an unreadable value cannot supply a number.
demo_numbers = demo_numbers.replace([float("inf"), float("-inf")], float("nan"))
display(pd.DataFrame({"Original": number_demo, "Interpreted number": demo_numbers}))


**Read the result:** 12.5 and -2 supply numbers. unknown and infinity cannot supply ordinary numbers. The original missing value is still missing. errors="coerce" changes unsuccessful conversions in the temporary result, not in the source.



**Pause and explain:** Why can -2 pass numerical readiness but later be Flagged as a fire size?

<details>
<summary>Suggested answer — open after discussing</summary>

It is a readable number, but it does not meet our positive-area expectation.

</details>


In [ ]:
# This separate table will carry ordinary numbers to Chapter 4 without rechecking
# the original text. Its source-row labels remain aligned with raw.
number_values = pd.DataFrame(index=raw.index)
number_values["fire_size_value"] = pd.to_numeric(raw["fire_size"], errors="coerce")
number_values["fire_size_value"] = number_values["fire_size_value"].replace(
    [float("inf"), float("-inf")], float("nan")
)
size_source_present = raw["fire_size"].notna()
size_has_number = number_values["fire_size_value"].notna()
size_readability_flag = size_source_present & ~size_has_number
size_readability_untested = ~size_source_present
check_size_readiness = summarise_check(
    "CHK07", "Size supplies an ordinary number", len(raw),
    size_readability_flag.sum(), size_readability_untested.sum()
)
evidence_size_readiness = raw.loc[
    size_readability_flag | size_readability_untested, ["fire_name", "fire_size"]
].copy()
evidence_size_readiness["Interpreted number"] = number_values.loc[evidence_size_readiness.index, "fire_size_value"]
evidence_size_readiness["Record outcome"] = "Flagged"
evidence_size_readiness.loc[size_readability_untested.loc[evidence_size_readiness.index], "Record outcome"] = "Untested"
display(pd.DataFrame([check_size_readiness]))
display(evidence_size_readiness.head(5))
save_check(
    check_size_readiness, evidence_size_readiness,
    finding="Some size values are absent or cannot supply an ordinary number.",
    effect="Q2 cannot compare or total those size values numerically.",
    next_action="Investigate the original values before choosing a documented correction or an analysis subset."
)


**Continue the same numerical block:** `number_values` is the two-column table being built here.
Its size column was just created. The next cell adds the interpreted year column and saves the
completed table. No variables from another teaching block are needed.


In [ ]:
number_values["fire_year_value"] = pd.to_numeric(raw["fire_year"], errors="coerce")
number_values["fire_year_value"] = number_values["fire_year_value"].replace(
    [float("inf"), float("-inf")], float("nan")
)
year_source_present = raw["fire_year"].notna()
year_has_number = number_values["fire_year_value"].notna()
year_readability_flag = year_source_present & ~year_has_number
year_readability_untested = ~year_source_present
check_year_readiness = summarise_check(
    "CHK08", "Year supplies an ordinary number", len(raw),
    year_readability_flag.sum(), year_readability_untested.sum()
)
evidence_year_readiness = raw.loc[
    year_readability_flag | year_readability_untested, ["fire_name", "fire_year"]
].copy()
evidence_year_readiness["Interpreted number"] = number_values.loc[evidence_year_readiness.index, "fire_year_value"]
evidence_year_readiness["Record outcome"] = "Flagged"
evidence_year_readiness.loc[year_readability_untested.loc[evidence_year_readiness.index], "Record outcome"] = "Untested"
display(pd.DataFrame([check_year_readiness]))
display(evidence_year_readiness.head(5))
save_check(
    check_year_readiness, evidence_year_readiness,
    finding="Some year values are absent or cannot supply an ordinary number.",
    effect="Q1 cannot place those values into meaningful numerical year comparisons.",
    next_action="Review the original values. A readable year will still need a whole-year and period check."
)
# Save once, here beside the variables' creation. Chapter 4 reloads these numbers.
number_values.to_csv("Outputs/number_values.csv", index=True, index_label="source_row")
print("Saved converted size and year values for Chapter 4.")


**Read the result:** For readability, an absent source value is Untested; a present unsuccessful conversion is Flagged. The file Outputs/number_values.csv contains all source rows and both interpreted columns.



**Pause and explain:** Does a readable year of 1998.5 meet our calendar-year expectation?

<details>
<summary>Suggested answer — open after discussing</summary>

It can supply a number but is not a whole calendar year. Chapter 4 checks that meaning separately.

</details>


## Block 5 · CHK09–CHK10 · Interpret date values

**Objective:** Separate absent dates from present values that cannot be interpreted as calendar dates.

**Expectation:** A present timestamp converts using YYYY-MM-DD HH:MM:SS. Missing input is Untested for readability; a present failed conversion is Flagged.

**Where it comes from:** The teaching CSV timestamp export format. The source field supplies a calendar date; ordering is a separate rule.

**Objects used here:** `date_demo` is a small example. `date_values` will be a separate two-column source-aligned table, saved within this block for Chapter 4.

**Method reference:** [Series()](https://pandas.pydata.org/docs/reference/api/pandas.Series.html) · [to_datetime()](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html) · [isna()](https://pandas.pydata.org/docs/reference/api/pandas.Series.isna.html) · [notna()](https://pandas.pydata.org/docs/reference/api/pandas.Series.notna.html) · [loc](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html) · [to_csv()](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_csv.html)


In [ ]:
date_demo = pd.Series(["2005-07-10 00:00:00", "2005-99-10 00:00:00", None])
demo_calendar_dates = pd.to_datetime(
    date_demo, format="%Y-%m-%d %H:%M:%S", errors="coerce"
)
display(pd.DataFrame({"Original": date_demo, "Interpreted date": demo_calendar_dates}))


**Read the result:** The valid timestamp becomes a date value. Month 99 cannot supply a calendar date and becomes NaT in the temporary result. The absent source also gives NaT, but its reason differs.



**Pause and explain:** Why do we need the original source presence as well as the converted result?

<details>
<summary>Suggested answer — open after discussing</summary>

Both missing input and failed conversion produce a missing result. Source presence tells us which case occurred.

</details>


In [ ]:
date_values = pd.DataFrame(index=raw.index)
date_values["discovery_date_value"] = pd.to_datetime(
    raw["discovery_date"], format="%Y-%m-%d %H:%M:%S", errors="coerce"
)
discovery_source_present = raw["discovery_date"].notna()
discovery_has_date = date_values["discovery_date_value"].notna()
discovery_readability_flag = discovery_source_present & ~discovery_has_date
discovery_readability_untested = ~discovery_source_present
check_discovery_readiness = summarise_check(
    "CHK09", "Discovery value supplies a date", len(raw),
    discovery_readability_flag.sum(), discovery_readability_untested.sum()
)
evidence_discovery_readiness = raw.loc[
    discovery_readability_flag | discovery_readability_untested, ["fire_name", "discovery_date"]
].copy()
evidence_discovery_readiness["Interpreted date"] = date_values.loc[evidence_discovery_readiness.index, "discovery_date_value"]
evidence_discovery_readiness["Record outcome"] = "Flagged"
evidence_discovery_readiness.loc[discovery_readability_untested.loc[evidence_discovery_readiness.index], "Record outcome"] = "Untested"
display(pd.DataFrame([check_discovery_readiness]))
display(evidence_discovery_readiness.head(5))
save_check(
    check_discovery_readiness, evidence_discovery_readiness,
    finding="Some discovery values are absent or cannot be interpreted using the CSV date format.",
    effect="Q3 cannot use those discovery endpoints; Q1 cannot extract a discovery year from them.",
    next_action="Review missing values and original date spellings before agreeing a correction."
)


**Continue the same date block:** `date_values` holds the converted discovery dates just created.
The next cell adds containment dates and saves both columns. This table changes our temporary
representation; it does not fill or correct dates in `raw`.


In [ ]:
date_values["contained_date_value"] = pd.to_datetime(
    raw["contained_date"], format="%Y-%m-%d %H:%M:%S", errors="coerce"
)
containment_source_present = raw["contained_date"].notna()
containment_has_date = date_values["contained_date_value"].notna()
containment_readability_flag = containment_source_present & ~containment_has_date
containment_readability_untested = ~containment_source_present
check_containment_readiness = summarise_check(
    "CHK10", "Containment value supplies a date", len(raw),
    containment_readability_flag.sum(), containment_readability_untested.sum()
)
evidence_containment_readiness = raw.loc[
    containment_readability_flag | containment_readability_untested, ["fire_name", "contained_date"]
].copy()
evidence_containment_readiness["Interpreted date"] = date_values.loc[evidence_containment_readiness.index, "contained_date_value"]
evidence_containment_readiness["Record outcome"] = "Flagged"
evidence_containment_readiness.loc[containment_readability_untested.loc[evidence_containment_readiness.index], "Record outcome"] = "Untested"
display(pd.DataFrame([check_containment_readiness]))
display(evidence_containment_readiness.head(5))
save_check(
    check_containment_readiness, evidence_containment_readiness,
    finding="Some containment values are absent or cannot be interpreted using the CSV date format.",
    effect="Q3 cannot use those endpoints. Passing readable values does not recover absent dates.",
    next_action="Seek reliable date recovery or define a duration subset. Investigate present conversion failures separately."
)
# CSV stores interpreted dates as text for transfer. Chapter 4 restores their
# date type while loading; it does not repeat checks on the original strings.
date_values.to_csv(
    "Outputs/date_values.csv", index=True, index_label="source_row",
    date_format="%Y-%m-%d %H:%M:%S"
)
print("Saved converted discovery and containment values for Chapter 4.")


**Read the result:** Both date columns are saved for all records. Failed and absent dates remain unavailable in the converted table, while the check and evidence files preserve why they are unavailable.



**Pause and explain:** Can a conversion success establish that containment is after discovery?

<details>
<summary>Suggested answer — open after discussing</summary>

No. A readable calendar date may still have the wrong relationship to another date.

</details>


## Block 6 · Apply and explain a familiar representation method

**Student activities:**

1. Predict what trimming does to `"   "`, `"Oak"` and `" OAK "`. Does it also change case?
2. Explain why `"Missing/Undefined"` is different from a missing cell.
3. Predict the readability outcomes for `"0"`, `"unknown"` and an absent size.
4. Explain why a value can pass readability and fail meaning.

**Suggested reasoning:** Trimming removes outside whitespace, so a space-only name becomes
empty; letter case needs a separate comparison method. Missing/Undefined is actual text
identifying unavailable cause information. Zero is readable, unknown is present but unreadable,
and an absent value cannot enter readability. Positive size and a whole year are meaning rules.

**Optional adaptation, without adding a new core check:** Use `value_counts()` to inspect
`fire_class`, then explain why observed labels alone do not prove that the labels are appropriate.
Detailed class boundaries belong to later work. The core evidence already demonstrates
missing, changed and unsuitable values.


## Block 7 · Review the outputs and prepare for meaning checks

**Objective:** Know exactly what Chapter 4 will load.

| Saved result | Contents | Created beside its code |
| --- | --- | --- |
| `Outputs/number_values.csv` | `source_row`, `fire_size_value`, `fire_year_value` | Block 4 |
| `Outputs/date_values.csv` | `source_row`, `discovery_date_value`, `contained_date_value` | Block 5 |
| `Outputs/checks/CHK04.csv` through `CHK10.csv` | One count summary for each check | Each check's own cell |
| Corresponding `Outputs/issues`, `Outputs/evidence` and `Outputs/previews` files | Written concerns, full original evidence and short balanced examples | Each check's own cell |

**File discipline:** Use the same unchanged `Data/wildfires.csv` throughout the sequence.
If you replace the source file, rerun Chapters 2–4 before generating the final report. Saved
outputs do not update merely because the source file changed.

**Chapter recap:** Present, readable and meaningful are different questions. Each needs its
own expectation and prerequisite. All temporary conversions and comparisons leave `raw` unchanged.
